# GEPA Skill Optimization with Evals

Optimize Claude Agent Skills using **DSPy's GEPA** optimizer with **evaluation test cases and LLM-as-judge assertions**.

## How it differs from the static-only notebook

| | Static-only | Eval-based (this notebook) |
|---|---|---|
| **Metric** | 100% static analysis | 40% static + 60% LLM-as-judge |
| **Scoring** | Filler removal, conciseness, code blocks, structure | Same static checks + assertion pass rate |
| **Evals** | None | Test cases with verifiable assertions |
| **Validation** | Metric score only | Per-assertion PASS/FAIL verdicts |

## Pipeline

1. **Setup** — Import dependencies, configure LLM
2. **Load Skill** — Parse skill directory
3. **Analyze** — Check against best practices
4. **Load / Generate Evals** — Load test cases, generate assertions
5. **Define Module** — DSPy signature + ChainOfThought
6. **Define Metric** — Hybrid: static analysis + LLM-as-judge
7. **Run GEPA** — Evolutionary prompt optimization
8. **Validate** — Final assertion pass rates
9. **Save** — Output optimized skill + benchmark

## 1. Setup

In [1]:
import json
import re
import os
import sys
from pathlib import Path

import dspy
from dspy import GEPA
from dotenv import load_dotenv

# Add repo root to path so we can import skillopt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from skillopt import SkillParser, SkillAnalyzer

load_dotenv()

parser = SkillParser()
analyzer = SkillAnalyzer()

# Configure DSPy with your LLM
lm = dspy.LM("openai/gpt-4o", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

print(f"Configured DSPy with: {lm.model}")

Configured DSPy with: openai/gpt-4o


## 2. Load Skill

Set `SKILL_PATH` to your skill directory and `EVALS_PATH` to your evals JSON (or `None` to auto-generate).

In [3]:
# ---- CONFIGURE THESE ----
SKILL_PATH = ROOT / "examples/example_2/dad-joke"
EVALS_PATH = None  # set to None to auto-generate
OUTPUT_DIR = None  # set a Path to save results, or leave None to skip saving
# --------------------------

skill = parser.parse_directory(SKILL_PATH)
skill_content = skill.main_file.raw_content

print(f"Loaded skill: {skill.name}")
print(f"  Lines: {skill.total_lines}")
print(f"  References: {len(skill.references)}")

Loaded skill: dad-joke
  Lines: 141
  References: 0


## 3. Analyze Skill

In [4]:
report = analyzer.analyze(skill)

errors = [i for i in report.issues if i.severity == "error"]
warnings = [i for i in report.issues if i.severity == "warning"]
suggestions = [i for i in report.issues if i.severity == "suggestion"]

print(f"Score: {report.score}/100")
print(f"Issues: {len(report.issues)} ({len(errors)} errors, {len(warnings)} warnings, {len(suggestions)} suggestions)")
print()
for issue in report.issues:
    icon = {"error": "[ERR]", "warning": "[WRN]", "suggestion": "[SUG]"}.get(issue.severity, "[?]")
    print(f"  {icon} {issue.category}: {issue.message}")

Score: 100/100
Issues: 2 (0 errors, 1 warnings, 1 suggestions)

  [SUG] workflow: Multi-step workflow without checklist
  [WRN] frontmatter: Description may not explain when to use the skill


## 4. Load / Generate Evals & Assertions

If `EVALS_PATH` points to a file, it loads the test cases from it. Otherwise, test cases are auto-generated from the skill content.

Either way, any test case missing `expectations` gets assertions auto-generated via an LLM call (3–6 per case, filtered for objectivity).

In [5]:
# DSPy signatures for generating eval cases and assertions

class GenerateEvalCases(dspy.Signature):
    """Generate evaluation test cases for a Claude Agent Skill.

    Create 3 realistic test prompts a user would actually say. Cover
    different aspects of the skill. Make prompts specific and detailed
    with context (file paths, names, scenarios), not generic requests.
    """
    skill_content: str = dspy.InputField(desc="The skill's SKILL.md content")
    skill_name: str = dspy.InputField(desc="Name of the skill")
    test_cases: str = dspy.OutputField(
        desc='JSON: {"evals": [{"id": 1, "name": "short-name", "prompt": "realistic user prompt", "expected_output": "description of success"}]}'
    )


class GenerateAssertions(dspy.Signature):
    """Generate verifiable assertions for a skill evaluation test case.

    Given a skill's content and an evaluation prompt, generate specific,
    objectively verifiable assertions that a correct response should satisfy.

    Good assertions:
    - \"The response includes the kubectl logs command\"
    - \"The response mentions checking pod events with kubectl describe\"

    Bad assertions (too subjective):
    - \"The output is well-formatted\"
    - \"The response is helpful\"
    """
    skill_content: str = dspy.InputField(desc="The SKILL.md content")
    eval_prompt: str = dspy.InputField(desc="The user's task prompt")
    expected_output: str = dspy.InputField(desc="Description of expected result")
    assertions: str = dspy.OutputField(
        desc='JSON array of 3-6 verifiable assertion strings, e.g. ["assertion 1", "assertion 2"]'
    )

print("Defined GenerateEvalCases and GenerateAssertions signatures")

Defined GenerateEvalCases and GenerateAssertions signatures


In [6]:
# Load or generate eval cases

if EVALS_PATH and Path(EVALS_PATH).exists():
    with open(EVALS_PATH) as f:
        evals_data = json.load(f)
    print(f"Loaded evals from: {EVALS_PATH}")
else:
    print("No evals file found — auto-generating from skill content...")
    generator = dspy.ChainOfThought(GenerateEvalCases)
    result = generator(skill_content=skill_content[:6000], skill_name=skill.name)
    try:
        evals_data = json.loads(result.test_cases)
        if "evals" not in evals_data:
            evals_data = {"evals": evals_data if isinstance(evals_data, list) else []}
    except json.JSONDecodeError:
        evals_data = {
            "evals": [{"id": 1, "name": "basic-usage",
                       "prompt": f"Help me with a typical {skill.name} task",
                       "expected_output": "Should provide relevant guidance", "expectations": []}]
        }
    evals_data["skill_name"] = skill.name
    print(f"  Generated {len(evals_data.get('evals', []))} eval case(s)")

eval_cases = evals_data.get("evals", [])
print(f"  Total eval cases: {len(eval_cases)}")
for ec in eval_cases:
    print(f"    [{ec.get('id')}] {ec.get('name', 'unnamed')}")

No evals file found — auto-generating from skill content...
  Generated 1 eval case(s)
  Total eval cases: 1
    [1] basic-usage


In [7]:
# Generate assertions for any eval case that doesn't already have them

assertion_gen = dspy.ChainOfThought(GenerateAssertions)

for eval_case in eval_cases:
    if eval_case.get("expectations"):
        continue

    print(f"Generating assertions for eval {eval_case.get('id')}: {eval_case.get('name')}...")
    result = assertion_gen(
        skill_content=skill_content[:6000],
        eval_prompt=eval_case["prompt"],
        expected_output=eval_case.get("expected_output", ""),
    )
    try:
        parsed = json.loads(result.assertions)
        eval_case["expectations"] = parsed if isinstance(parsed, list) else []
    except json.JSONDecodeError:
        eval_case["expectations"] = [
            l.strip().lstrip("-*").strip()
            for l in result.assertions.strip().split("\n")
            if l.strip() and len(l.strip()) > 10
        ]

# Summary
total_assertions = sum(len(ec.get("expectations", [])) for ec in eval_cases)
print(f"\nEval cases with assertions ({total_assertions} total):")
for ec in eval_cases:
    exps = ec.get("expectations", [])
    print(f"  [{ec.get('id')}] {ec.get('name', 'unnamed')}: {len(exps)} assertion(s)")
    for a in exps:
        print(f"      - {a}")

Generating assertions for eval 1: basic-usage...

Eval cases with assertions (5 total):
  [1] basic-usage: 5 assertion(s)
      - The response includes a joke that uses a pun
      - The joke is short and does not require a lengthy setup
      - The joke is wholesome and suitable for all ages
      - The joke format is one of those specified: question-and-answer, setup-punchline, one-liner, observational, or self-referential
      - The response does not include any explanation of the joke unless asked by the user


## 5. Define DSPy Module

The signature's docstring becomes the initial prompt instruction. GEPA will evolve it over multiple iterations to maximize the metric.

In [8]:
class OptimizeSkillWithEvals(dspy.Signature):
    """Optimize a Claude Agent Skill to be effective, concise, and satisfy all evaluation criteria.

    Given the original skill content and evaluation test cases with assertions,
    rewrite the skill so that:

    1. HANDLE ALL TEST CASES: The optimized skill must contain the knowledge,
       commands, workflows, and guidance needed for an agent to produce responses
       satisfying every assertion in every test case.

    2. BE CONCISE: Remove filler phrases (\"make sure to\", \"ensure that\",
       \"don't forget to\"), verbose explanations of concepts the model already
       knows (what YAML/JSON/PDF is), and redundant or unrelated content.

    3. PRESERVE STRUCTURE: Keep frontmatter (--- name/description ---),
       section headers (##), and essential code blocks (```bash, ```python).

    4. CONSOLIDATE: Merge similar commands using placeholders (e.g., -n <namespace>).

    The output must be a complete, valid SKILL.md file starting with --- frontmatter.
    """
    original_skill: str = dspy.InputField(desc="Original SKILL.md content to optimize")
    eval_cases: str = dspy.InputField(
        desc="Evaluation test cases with prompts and assertions the optimized skill must satisfy"
    )
    optimized_skill: str = dspy.OutputField(
        desc="Complete optimized SKILL.md content that is concise and enables handling all test cases"
    )


class SkillOptimizerWithEvalsModule(dspy.Module):
    """DSPy module that optimizes skills guided by eval assertions."""

    def __init__(self):
        super().__init__()
        self.optimize = dspy.ChainOfThought(OptimizeSkillWithEvals)

    def forward(self, original_skill: str, eval_cases: str) -> str:
        result = self.optimize(
            original_skill=original_skill,
            eval_cases=eval_cases,
        )
        return result.optimized_skill


optimizer_module = SkillOptimizerWithEvalsModule()
print("Created SkillOptimizerWithEvalsModule")

Created SkillOptimizerWithEvalsModule


## 6. Define Metric

The hybrid metric combines:
- **40% static analysis** — filler removal, conciseness, code-block preservation, structure
- **60% assertion satisfaction** — LLM-as-judge evaluates whether the optimized skill enables handling each test case

In [9]:
# LLM-as-Judge signature

class JudgeAssertions(dspy.Signature):
    """Judge whether an optimized skill would guide an agent to satisfy assertions.

    For each assertion:
    - PASS: The skill clearly provides the knowledge, commands, or workflow
      needed to produce output satisfying this assertion.
    - FAIL: The skill lacks the necessary information, commands, or guidance.

    Be strict: the skill must contain actionable content (specific commands,
    steps, examples) that directly enables satisfying the assertion.
    """
    optimized_skill: str = dspy.InputField(desc="The optimized skill content being evaluated")
    eval_prompt: str = dspy.InputField(desc="The user's task prompt")
    assertions: str = dspy.InputField(desc="JSON array of assertions to evaluate")
    verdicts: str = dspy.OutputField(
        desc='JSON array of {"assertion": "...", "passed": true/false, "reason": "brief evidence"}'
    )

judge_module = dspy.ChainOfThought(JudgeAssertions)
print("Created JudgeAssertions module")

Created JudgeAssertions module


In [10]:
def count_code_blocks(content: str) -> int:
    return len(re.findall(r'```', content)) // 2


def static_analysis_score(optimized: str, original: str) -> float:
    """Static analysis score (0.0-1.0).

    30% filler removal, 25% conciseness, 25% code blocks, 20% structure.
    """
    score = 0.0

    # Filler removal (30%)
    filler_phrases = [
        'make sure to', 'ensure that', "don't forget to", 'remember to',
        'it is important to', 'please note that', 'keep in mind',
        'you should', 'you need to', 'you must',
    ]
    filler_count = sum(1 for p in filler_phrases if p.lower() in optimized.lower())
    score += 0.30 * max(0, 1 - (filler_count / 5))

    # Conciseness (25%) — sweet spot 30-70% reduction
    orig_len = len(original) if original else 1
    opt_len = len(optimized)
    if opt_len < orig_len:
        ratio = 1 - (opt_len / orig_len)
        if ratio < 0.3:
            score += 0.25 * (ratio / 0.3)
        elif ratio <= 0.7:
            score += 0.25
        else:
            score += 0.25 * max(0, 1 - (ratio - 0.7) * 3)

    # Code block preservation (25%)
    orig_blocks = count_code_blocks(original)
    opt_blocks = count_code_blocks(optimized)
    if orig_blocks > 0:
        block_ratio = opt_blocks / orig_blocks
        score += 0.25 * (min(1.0, block_ratio) if block_ratio >= 0.5 else block_ratio * 0.5)
    else:
        score += 0.25

    # Structure (20%)
    has_frontmatter = optimized.strip().startswith('---')
    has_sections = '## ' in optimized
    has_code = '```' in optimized
    score += 0.20 * ((0.4 if has_frontmatter else 0) + (0.3 if has_sections else 0) + (0.3 if has_code else 0))

    return score


def judge_skill_against_assertions(optimized_skill, eval_cases_list, judge):
    """LLM-as-judge: evaluate skill against all assertions. Returns (score, details)."""
    total_assertions = 0
    total_passed = 0
    all_results = []

    for ec in eval_cases_list:
        assertions = ec.get("expectations", [])
        if not assertions:
            continue

        result = judge(
            optimized_skill=optimized_skill,
            eval_prompt=ec["prompt"],
            assertions=json.dumps(assertions),
        )

        try:
            verdicts = json.loads(result.verdicts)
            passed = sum(1 for v in verdicts if v.get("passed", False)) if isinstance(verdicts, list) else 0
            total = len(verdicts) if isinstance(verdicts, list) else len(assertions)
        except (json.JSONDecodeError, TypeError):
            text = str(result.verdicts).upper()
            passed = max(0, text.count("PASS") - text.count("NOT PASS"))
            total = len(assertions)
            verdicts = []

        total_assertions += total
        total_passed += max(0, passed)
        all_results.append({
            "eval_id": ec.get("id", 0), "eval_name": ec.get("name", ""),
            "passed": max(0, passed), "total": total,
            "pass_rate": max(0, passed) / total if total > 0 else 0,
            "verdicts": verdicts,
        })

    return (total_passed / total_assertions if total_assertions > 0 else 0), all_results


def create_eval_metric(eval_cases_list, judge):
    """Create metric closure: 40% static + 60% assertion satisfaction."""

    def metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
        optimized = pred.optimized_skill if hasattr(pred, 'optimized_skill') else str(pred)
        original = gold.original_skill if hasattr(gold, 'original_skill') else ""

        static = static_analysis_score(optimized, original)
        assertion_score, _ = judge_skill_against_assertions(optimized, eval_cases_list, judge)

        return 0.40 * static + 0.60 * assertion_score

    return metric


metric_fn = create_eval_metric(eval_cases, judge_module)
print("Metric ready: 40% static + 60% LLM-as-judge")

Metric ready: 40% static + 60% LLM-as-judge


### Prepare Training Data

In [11]:
# Format eval cases as text for the module input
def format_eval_cases(cases):
    lines = []
    for ec in cases:
        lines.append(f"## Test Case {ec.get('id', '?')}: {ec.get('name', 'unnamed')}")
        lines.append(f"Prompt: {ec['prompt']}")
        lines.append(f"Expected: {ec.get('expected_output', 'N/A')}")
        for a in ec.get("expectations", []):
            lines.append(f"  - {a}")
        lines.append("")
    return "\n".join(lines)

eval_cases_str = format_eval_cases(eval_cases)

# Training set: the skill + eval cases
trainset = [
    dspy.Example(
        original_skill=skill_content,
        eval_cases=eval_cases_str,
    ).with_inputs("original_skill", "eval_cases")
]

print(f"Training set: {len(trainset)} example(s)")
print(f"Eval cases: {len(eval_cases)}")
print(f"Total assertions: {total_assertions}")

Training set: 1 example(s)
Eval cases: 1
Total assertions: 5


## 7. Run GEPA Optimization

GEPA evolves the module's prompt instructions over multiple iterations. Each candidate is scored by the hybrid metric. The reflection LM proposes improvements; only better-scoring candidates are kept.

In [12]:
# Reflection LM (higher temperature for creative proposals)
reflection_lm = dspy.LM(
    "openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=4096,
)

MAX_EVALS = 10  # number of GEPA iterations

gepa = GEPA(
    metric=metric_fn,
    reflection_lm=reflection_lm,
    max_full_evals=MAX_EVALS,
    num_threads=1,
    reflection_minibatch_size=min(3, len(trainset)),
    skip_perfect_score=True,
)

print(f"Running GEPA ({MAX_EVALS} iterations)...")
print(f"  Metric: 40% static + 60% assertion ({total_assertions} assertions)")
print()

optimized_module = gepa.compile(
    student=optimizer_module,
    trainset=trainset,
)

print("\nGEPA optimization complete")

2026/04/16 00:36:10 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 10 metric calls of the program. This amounts to 10.00 full evals on the train set.
2026/04/16 00:36:10 WARNING dspy.teleprompt.gepa.gepa: No valset provided; Using trainset as valset. This is useful as an inference-time scaling strategy where you want GEPA to find the best solutions for the provided tasks in the trainset, as it makes GEPA overfit prompts to the provided trainset. In order to ensure generalization and perform well on unseen tasks, please provide separate trainset and valset. Provide the smallest valset that is just large enough to match the downstream task distribution, while keeping trainset as large as possible.
2026/04/16 00:36:10 INFO dspy.teleprompt.gepa.gepa: Using 1 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just l

Running GEPA (10 iterations)...
  Metric: 40% static + 60% assertion (5 assertions)



GEPA Optimization:   0%|          | 0/10 [00:00<?, ?rollouts/s]2026/04/16 00:36:20 INFO dspy.evaluate.evaluate: Average Metric: 0.976 / 1 (97.6%)
2026/04/16 00:36:20 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.976 over 1 / 1 examples
GEPA Optimization:  10%|█         | 1/10 [00:10<01:31, 10.13s/rollouts]2026/04/16 00:36:20 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.976


Average Metric: 0.98 / 1 (97.6%): 100%|██████████| 1/1 [00:00<00:00, 23.23it/s]

2026/04/16 00:36:20 INFO dspy.evaluate.evaluate: Average Metric: 0.976 / 1 (97.6%)


...

Average Metric: 0.98 / 1 (97.6%): 100%|██████████| 1/1 [00:00<00:00, 42.15it/s]


2026/04/16 00:36:37 INFO dspy.evaluate.evaluate: Average Metric: 0.976 / 1 (97.6%)
2026/04/16 00:36:37 INFO dspy.teleprompt.gepa.gepa: Iteration 4: Proposed new text for optimize.predict: Optimize a Claude Agent Skill by improving its effectiveness and conciseness while ensuring it meets evaluation criteria.

Instructions:
1. **Test Case Handling:** Ensure the rewritten skill comprehensively addresses every assertion outlined in the provided test cases. The skill must include essential knowledge, commands, workflows, and guidance to help an agent provide responses that satisfy all evaluation criteria.

2. **Conciseness:** Remove any unnecessary filler phrases and verbose explanations of concepts that the agent already understands. Only include essential and relevant information, focusing on brevity without losing clarity.

3. **Preserve Structure:** Maintain the original structure of the SKILL.md, including frontmatter (--- name/description ---), section headers (##), and necessary cod

Average Metric: 0.98 / 1 (97.6%): 100%|██████████| 1/1 [00:00<00:00, 44.88it/s]

2026/04/16 00:36:37 INFO dspy.evaluate.evaluate: Average Metric: 0.976 / 1 (97.6%)
2026/04/16 00:36:37 INFO dspy.teleprompt.gepa.gepa: Iteration 5: Proposed new text for optimize.predict: Optimize a Claude Agent Skill by improving its effectiveness and conciseness while ensuring it meets evaluation criteria.

Instructions:
1. **Test Case Handling:** Ensure the rewritten skill comprehensively addresses every assertion outlined in the provided test cases. The skill must include essential knowledge, commands, workflows, and guidance to help an agent provide responses that satisfy all evaluation criteria.

2. **Conciseness:** Remove any unnecessary filler phrases and verbose explanations of concepts that the agent already understands. Only include essential and relevant information, focusing on brevity without losing clarity.

3. **Preserve Structure:** Maintain the original structure of the SKILL.md, including frontmatter (--- name/description ---), section headers (##), and necessary cod



GEPA optimization complete


## 8. Apply & Validate

Run the optimized module on the full skill, then validate with the LLM judge.

In [13]:
# Apply the optimized module
result = optimized_module(
    original_skill=skill_content,
    eval_cases=eval_cases_str,
)

optimized_content = result.optimized_skill if hasattr(result, 'optimized_skill') else str(result)

# Metrics
original_len = len(skill_content)
optimized_len = len(optimized_content)
reduction_pct = ((original_len - optimized_len) / original_len * 100) if original_len > 0 else 0
orig_blocks = count_code_blocks(skill_content)
opt_blocks = count_code_blocks(optimized_content)

print(f"Original:  {original_len:,} chars, {orig_blocks} code blocks")
print(f"Optimized: {optimized_len:,} chars, {opt_blocks} code blocks")
print(f"Reduction: {original_len - optimized_len:,} chars ({reduction_pct:.1f}%)")

Original:  4,466 chars, 0 code blocks
Optimized: 1,807 chars, 0 code blocks
Reduction: 2,659 chars (59.5%)


In [14]:
# Final assertion validation
final_assertion_score, detailed_results = judge_skill_against_assertions(
    optimized_content, eval_cases, judge_module
)

static = static_analysis_score(optimized_content, skill_content)
combined = 0.40 * static + 0.60 * final_assertion_score

print(f"Scores:")
print(f"  Static analysis:     {static:.3f}")
print(f"  Assertion pass rate: {final_assertion_score:.3f}")
print(f"  Combined (40/60):    {combined:.3f}")
print()
print("Assertion results by eval:")
for r in detailed_results:
    status = "PASS" if r['pass_rate'] == 1.0 else "PARTIAL" if r['pass_rate'] > 0 else "FAIL"
    print(f"  [{r['eval_id']}] {r['eval_name']}: {r['passed']}/{r['total']} ({r['pass_rate']:.0%}) {status}")
    if isinstance(r.get('verdicts'), list):
        for v in r['verdicts']:
            icon = "\u2713" if v.get('passed') else "\u2717"
            print(f"      {icon} {v.get('assertion', '')}")

Scores:
  Static analysis:     0.940
  Assertion pass rate: 1.000
  Combined (40/60):    0.976

Assertion results by eval:
  [1] basic-usage: 5/5 (100%) PASS
      ✓ The response includes a joke that uses a pun
      ✓ The joke is short and does not require a lengthy setup
      ✓ The joke is wholesome and suitable for all ages
      ✓ The joke format is one of those specified: question-and-answer, setup-punchline, one-liner, observational, or self-referential
      ✓ The response does not include any explanation of the joke unless asked by the user


In [15]:
# Preview the optimized skill
print("Optimized SKILL.md preview:")
print("=" * 60)
print(optimized_content[:3000])
if len(optimized_content) > 3000:
    print("\n... (truncated)")

Optimized SKILL.md preview:
---
name: dad-joke
description: Generate short, pun-driven, wholesome dad jokes.
---

# Dad Joke Generator

As a dad joke expert, your goal is to produce jokes in the dad joke tradition: short, pun-driven, wholesome, and aimed at eliciting groans.

## When to use

Use this skill when the user requests a dad joke or a similar type of humor, like corny or clean jokes.

## Characteristics of Dad Jokes

- **Mandatory Puns:** Every dad joke must include a pun, relying on wordplay or double meanings.
- **Simplicity and Obviousness:** The pun should be immediately obvious, leading to a groan.
- **Wholesome:** Always appropriate for all audiences, without any offensive content.
- **Concise:** Limited to one or two sentences.
- **Enthusiastic Delivery:** Present with genuine humor as though it's the funniest joke ever.

## Joke Formats

Use and vary these formats to keep jokes engaging:

- **Question-and-Answer:** A setup question with a pun in the answer.
- **Setup-

## 9. Save Results

Set `OUTPUT_DIR` at the top of the notebook to save the optimized skill and benchmark.

In [16]:
if OUTPUT_DIR:
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    # Save optimized SKILL.md
    (output_path / "SKILL.md").write_text(optimized_content)

    # Save benchmark results
    benchmark = {
        "metadata": {
            "skill_name": skill.name,
            "model": lm.model,
            "optimizer": "GEPA",
            "max_evals": MAX_EVALS,
            "metric_weights": {"static": 0.40, "assertions": 0.60},
        },
        "scores": {
            "static_analysis": round(static, 4),
            "assertion_pass_rate": round(final_assertion_score, 4),
            "combined": round(combined, 4),
        },
        "metrics": {
            "original_chars": original_len,
            "optimized_chars": optimized_len,
            "reduction_percent": round(reduction_pct, 1),
            "original_code_blocks": orig_blocks,
            "optimized_code_blocks": opt_blocks,
        },
        "assertion_results": detailed_results,
        "evals": eval_cases,
    }

    with open(output_path / "benchmark.json", "w") as f:
        json.dump(benchmark, f, indent=2)

    print(f"Saved to: {output_path}")
    print(f"  SKILL.md       \u2014 optimized skill")
    print(f"  benchmark.json \u2014 scores + assertion verdicts")
else:
    print("OUTPUT_DIR not set \u2014 skipping save. Set it in cell 2 to enable.")

OUTPUT_DIR not set — skipping save. Set it in cell 2 to enable.
